In [25]:
import msal
import webbrowser

In [26]:
import requests
import pandas as pd

In [27]:
class PowerBIInteractiveAuth:
    def __init__(self, tenant_id="common", client_id=None):
        """
        tenant_id: Use 'common' for multi-tenant or your Rio Tinto tenant ID
        client_id: Use Microsoft's public client ID if you don't have your own
        """
        self.tenant_id = tenant_id
        # Microsoft's public Power BI client ID (no registration needed)
        self.client_id = client_id or "ea0616ba-638b-4df5-95b9-636659ae5121"
        self.authority = f"https://login.microsoftonline.com/{tenant_id}"
        self.scope = ["https://analysis.windows.net/powerbi/api/.default"]
        
    def get_access_token(self):
        """Authenticate using interactive browser login"""
        app = msal.PublicClientApplication(
            self.client_id,
            authority=self.authority
        )
        
        # Try to get token from cache first
        accounts = app.get_accounts()
        if accounts:
            result = app.acquire_token_silent(self.scope, account=accounts[0])
            if result and "access_token" in result:
                print("Token acquired from cache")
                return result["access_token"]
        
        # Interactive login via browser
        print("Opening browser for authentication...")
        result = app.acquire_token_interactive(
            scopes=self.scope,
            prompt="select_account"  # Always show account picker
        )
        
        if "access_token" in result:
            print(f"Authenticated as: {result.get('id_token_claims', {}).get('preferred_username', 'Unknown')}")
            return result["access_token"]
        else:
            raise Exception(f"Authentication failed: {result.get('error_description')}")

In [28]:
class PowerBIClient:
    def __init__(self, access_token):
        self.base_url = "https://api.powerbi.com/v1.0/myorg"
        self.headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        }
    
    def get_workspaces(self):
        """Get all workspaces (groups) you have access to"""
        url = f"{self.base_url}/groups"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()['value']
    
    def get_workspace_by_name(self, workspace_name):
        """Find workspace by name"""
        workspaces = self.get_workspaces()
        for ws in workspaces:
            if ws['name'] == workspace_name:
                return ws
        return None
    
    def get_datasets(self, workspace_id):
        """Get all datasets in a workspace"""
        url = f"{self.base_url}/groups/{workspace_id}/datasets"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()['value']
    
    def get_dataset_refresh_history(self, workspace_id, dataset_id):
        """Get refresh history for a dataset"""
        url = f"{self.base_url}/groups/{workspace_id}/datasets/{dataset_id}/refreshes"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()['value']
    
    def get_dataset_users(self, dataset_id):
        """Get users with access to dataset (Admin API)"""
        url = f"{self.base_url}/admin/datasets/{dataset_id}/users"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()['value']
    
    def trigger_dataset_refresh(self, workspace_id, dataset_id):
        """Trigger a dataset refresh"""
        url = f"{self.base_url}/groups/{workspace_id}/datasets/{dataset_id}/refreshes"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.status_code == 202

In [29]:
import requests
import pandas as pd
from datetime import datetime, timedelta

class PowerBIDatasetUsage(PowerBIClient):
    
    def get_dataset_users_with_access(self, dataset_id):
        """Get list of users who have access to a dataset (requires admin rights)"""
        url = f"{self.base_url}/admin/datasets/{dataset_id}/users"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('value', [])
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 401:
                print("⚠️ Admin rights required for this endpoint")
            return []
    
    def get_workspace_users(self, workspace_id):
        """Get all users with access to a workspace"""
        url = f"{self.base_url}/groups/{workspace_id}/users"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('value', [])
        except Exception as e:
            print(f"Error getting workspace users: {e}")
            return []
    
    def get_activity_events(self, start_date, end_date=None):
        """
        Get Power BI activity events (requires admin rights)
        start_date: 'YYYY-MM-DD' format
        end_date: 'YYYY-MM-DD' format (defaults to today)
        """
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        
        url = f"{self.base_url}/admin/activityevents"
        params = {
            'startDateTime': f"{start_date}T00:00:00.000Z",
            'endDateTime': f"{end_date}T23:59:59.999Z"
        }
        
        try:
            response = requests.get(url, headers=self.headers, params=params)
            response.raise_for_status()
            
            # Activity logs can be large, handle continuation token
            activities = []
            data = response.json()
            
            if 'activityEventEntities' in data:
                activities.extend(data['activityEventEntities'])
            
            # Handle pagination if there's a continuation token
            while 'continuationToken' in data:
                continuation_url = data['continuationUri']
                response = requests.get(continuation_url, headers=self.headers)
                response.raise_for_status()
                data = response.json()
                if 'activityEventEntities' in data:
                    activities.extend(data['activityEventEntities'])
            
            return activities
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 401:
                print("⚠️ Admin rights required for activity logs")
            elif e.response.status_code == 403:
                print("⚠️ Forbidden - check admin consent for activity logs")
            return []
    
    def get_dataset_usage_from_activities(self, dataset_id, days_back=30):
        """
        Extract dataset usage by user from activity logs
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        
        print(f"Fetching activity logs from {start_date.date()} to {end_date.date()}...")
        activities = self.get_activity_events(
            start_date.strftime('%Y-%m-%d'),
            end_date.strftime('%Y-%m-%d')
        )
        
        # Filter dataset-related activities
        dataset_activities = [
            act for act in activities 
            if act.get('DatasetId') == dataset_id or 
               act.get('DatasetName')  # Some activities use DatasetName
        ]
        
        return dataset_activities
    
    def analyze_dataset_usage(self, workspace_id, dataset_id, days_back=30):
        """
        Comprehensive dataset usage analysis
        Returns: DataFrame with user activity summary
        """
        print(f"\n{'='*60}")
        print(f"DATASET USAGE ANALYSIS")
        print(f"{'='*60}\n")
        
        # Get dataset info
        datasets = self.get_datasets(workspace_id)
        dataset = next((d for d in datasets if d['id'] == dataset_id), None)
        
        if not dataset:
            print("❌ Dataset not found")
            return None
        
        print(f"📊 Dataset: {dataset.get('name', 'Unknown')}")
        print(f"🆔 Dataset ID: {dataset_id}")
        print(f"📅 Analysis Period: Last {days_back} days\n")
        
        # 1. Get users with access
        print("1️⃣ Fetching users with access...")
        dataset_users = self.get_dataset_users_with_access(dataset_id)
        workspace_users = self.get_workspace_users(workspace_id)
        
        if dataset_users:
            df_access = pd.DataFrame(dataset_users)
            print(f"   ✓ Found {len(dataset_users)} users with dataset access")
        elif workspace_users:
            df_access = pd.DataFrame(workspace_users)
            print(f"   ✓ Found {len(workspace_users)} workspace users (dataset-level access requires admin)")
        else:
            df_access = pd.DataFrame()
            print("   ⚠️ Could not retrieve user access list (may require admin rights)")
        
        # 2. Get activity logs
        print("\n2️⃣ Fetching activity logs...")
        activities = self.get_dataset_usage_from_activities(dataset_id, days_back)
        
        if activities:
            print(f"   ✓ Found {len(activities)} activity events")
            
            # Create usage summary
            usage_data = []
            for act in activities:
                usage_data.append({
                    'UserId': act.get('UserId', 'Unknown'),
                    'UserEmail': act.get('UserKey', act.get('UserId', 'Unknown')),
                    'Activity': act.get('Activity', 'Unknown'),
                    'Timestamp': act.get('CreationTime', ''),
                    'Operation': act.get('Operation', 'Unknown'),
                    'WorkspaceName': act.get('WorkSpaceName', ''),
                })
            
            df_usage = pd.DataFrame(usage_data)
            
            # Aggregate by user
            df_summary = df_usage.groupby(['UserId', 'UserEmail']).agg({
                'Activity': 'count',
                'Timestamp': ['min', 'max']
            }).reset_index()
            
            df_summary.columns = ['UserId', 'UserEmail', 'TotalActivities', 'FirstAccess', 'LastAccess']
            df_summary = df_summary.sort_values('TotalActivities', ascending=False)
            
            return {
                'dataset_info': dataset,
                'users_with_access': df_access,
                'usage_summary': df_summary,
                'detailed_activities': df_usage
            }
        else:
            print("   ⚠️ No activity logs found (may require admin rights)")
            return {
                'dataset_info': dataset,
                'users_with_access': df_access,
                'usage_summary': pd.DataFrame(),
                'detailed_activities': pd.DataFrame()
            }
    
    def get_dataset_dependent_reports(self, workspace_id, dataset_id):
        """Get reports that use this dataset"""
        url = f"{self.base_url}/groups/{workspace_id}/datasets/{dataset_id}/dependentReports"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('value', [])
        except Exception as e:
            print(f"Error getting dependent reports: {e}")
            return []

In [30]:
client_id = "ea0616ba-638b-4df5-95b9-636659ae5121"

In [31]:
auth = PowerBIInteractiveAuth(
    tenant_id="common",  # Or use your Rio Tinto tenant ID
    client_id=client_id
)

In [32]:
access_token = auth.get_access_token()

Opening browser for authentication...
Authenticated as: ByambadorjMe@riotinto.com


In [33]:
pbi = PowerBIClient(access_token)

In [34]:
workspaces = pbi.get_workspaces()

In [35]:
prime_workspace = {
  'id': 'da007200-a034-4223-82af-8752eae843bb',
  'isReadOnly': False,
  'isOnDedicatedCapacity': True,
  'capacityId': '3FECB0D8-3030-485F-A653-EACFCB9149B2',
  'defaultDatasetStorageFormat': 'Small',
  'type': 'Workspace',
  'name': 'PRiME Dataset [Production]'
}

In [36]:
datasets = pbi.get_datasets(prime_workspace['id'])

In [37]:
df_datasets = pd.DataFrame(datasets)

In [38]:
print(df_datasets[['id', 'name', 'isRefreshable']])

                                     id  \
0  801ad8b1-4294-41c6-9dfc-0cdd04bb4f91   
1  418ae8f1-be34-4eff-9c3e-39fb37ef3113   
2  23ebecb5-fb18-46ec-b123-a06f1774f43e   
3  451084fc-1fb3-4a9e-88c4-deacab6d5568   
4  5be3751d-d0d7-439a-a90c-becbc762cae8   
5  cb641537-86f0-416f-a56b-bda9e1883744   
6  3c2a7b4d-5d09-4d24-84fa-7ff53898e318   
7  c49cd9b7-1ea0-444e-aebe-975cf28e75dd   
8  e586baaf-43be-4f8b-8a2d-c2ba03bb3976   
9  5b31e8bd-0087-4835-adf5-65902ab7b96b   

                                    name  isRefreshable  
0                  PRiME Reporting Unity           True  
1       PRiME EE Drilling Analysis Unity           True  
2  PRiME EE Drilling Cost Analysis Unity           True  
3                 PRIME Processing Unity           True  
4   PRIME Processing TUM Assurance Unity           True  
5           PRiME Asset Management Unity           True  
6             PRiME Surface Mining Unity           True  
7           PRIME HME Time & Cycle Unity           True  
8   

In [39]:
dataset_id = datasets[0]['id']
refresh_history = pbi.get_dataset_refresh_history(prime_workspace['id'], dataset_id)
df_refresh = pd.DataFrame(refresh_history)
print(f"\nRefresh history for {datasets[0]['name']}:")
print(df_refresh[['startTime', 'endTime', 'status']])


Refresh history for PRiME Reporting Unity:
                   startTime                   endTime     status
0    2026-01-18T18:45:28.48Z   2026-01-18T19:00:16.36Z  Completed
1   2026-01-17T19:41:03.123Z   2026-01-17T19:58:47.62Z  Completed
2   2026-01-16T18:40:22.117Z  2026-01-16T18:57:11.933Z  Completed
3   2026-01-16T05:39:21.357Z   2026-01-16T05:39:38.59Z  Completed
4   2026-01-16T05:37:25.657Z   2026-01-16T05:37:42.86Z  Completed
5   2026-01-16T05:33:39.193Z  2026-01-16T05:34:01.473Z  Completed
6   2026-01-16T05:31:35.337Z  2026-01-16T05:31:52.497Z  Completed
7   2026-01-16T05:28:44.867Z  2026-01-16T05:29:09.767Z  Completed
8   2026-01-16T04:13:35.037Z  2026-01-16T04:13:49.967Z  Completed
9    2026-01-16T03:37:06.63Z  2026-01-16T03:37:32.893Z  Completed
10    2026-01-16T01:45:20.6Z  2026-01-16T01:46:32.137Z  Completed
11  2026-01-16T01:44:46.227Z  2026-01-16T01:45:14.483Z  Completed
12  2026-01-16T01:43:59.537Z  2026-01-16T01:44:41.563Z  Completed
13  2026-01-16T01:42:46.733Z  20

In [40]:
pbi.get_dataset_users(prime_workspace['id'])

HTTPError: 401 Client Error: Unauthorized for url: https://api.powerbi.com/v1.0/myorg/admin/datasets/da007200-a034-4223-82af-8752eae843bb/users